In [ ]:
import xgboost as xgb 
import pandas as pd 
import sklearn as sk 
import re 
import matplotlib.pyplot as plt
import seaborn as sns 
import numpy as np
from matplotlib.ticker import MaxNLocator
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import KFold 
from sklearn.model_selection import GridSearchCV 
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from sklearn.metrics import accuracy_score 
from scipy.linalg import sqrtm
from sklearn.utils import shuffle
from matplotlib.ticker import LogLocator, NullFormatter
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics import confusion_matrix
%matplotlib inline
from matplotlib.collections import LineCollection
from sklearn import metrics

In [ ]:
#col = ['MissingET_MET_0', 'photon_energy', 'photon_theta_cos']
col = ['photon_energy', 'photon_theta_cos']
def readcsv(file_path, col = col):
    # 读取CSV文件
    data = pd.read_csv(file_path, usecols=col)
    # 确保列顺序与col一致
    data = data[col]
    return data

In [ ]:
a1MeV = readcsv('predict_data/select_data/m_a = 1   MeV.csv')
a10MeV = readcsv('predict_data/select_data/m_a = 10  MeV.csv')
a100MeV = readcsv('predict_data/select_data/m_a = 100 MeV.csv')
a1GeV = readcsv('predict_data/select_data/m_a = 1   GeV.csv')
a10GeV = readcsv('predict_data/select_data/m_a = 10  GeV.csv')
aME = readcsv('predict_data/select_data/Mono-gamma.csv')

In [ ]:
all_data = pd.concat([a1MeV,a10MeV,a100MeV,a1GeV,a10GeV,aME], axis = 0)
a1GeV.shape

In [ ]:
X = all_data[[ 'photon_energy', 'photon_theta_cos']]
#Y = all_data['label']

In [ ]:
model = xgb.XGBClassifier()
model.load_model('./clf_ALP_6_300_2')

# 倒入模型

In [ ]:
a1GeV.columns

In [ ]:
a1MeV_pre = model.predict_proba(a1MeV)[:,1:2]
a10MeV_pre = model.predict_proba(a10MeV)[:,1:2]
a100MeV_pre = model.predict_proba(a100MeV)[:,1:2]
a1GeV_pre = model.predict_proba(a1GeV)[:,1:2]
a10GeV_pre = model.predict_proba(a10GeV)[:,1:2]
aME_pre = model.predict_proba(aME)[:,1:2]

In [ ]:
eff_a1MeV = np.array([p for p in a1MeV_pre])
eff_a10MeV = np.array([p for p in a10MeV_pre])
eff_a100MeV = np.array([p for p in a100MeV_pre])
eff_a1GeV = np.array([p for p in a1GeV_pre])
eff_a10GeV = np.array([p for p in a10GeV_pre])
eff_aME = np.array([p for p in aME_pre])

In [ ]:
custom_cs_list = [50.21, 50.21, 50.209, 50.191,  48.42, 14]
pb_fb = 10**(3)
f_custom_cs_list_1000TeV = 1/(1000*1000)
P_ME_L_gt_450cm_list = [1,1,1,0.999986755,0.047942845]

cross_section_a1MeV = custom_cs_list[0]*pb_fb*f_custom_cs_list_1000TeV *P_ME_L_gt_450cm_list[0]
cross_section_a10MeV = custom_cs_list[1]*pb_fb*f_custom_cs_list_1000TeV *P_ME_L_gt_450cm_list[1]
cross_section_a100MeV = custom_cs_list[2]*pb_fb*f_custom_cs_list_1000TeV *P_ME_L_gt_450cm_list[2]
cross_section_a1GeV = custom_cs_list[3]*pb_fb*f_custom_cs_list_1000TeV *P_ME_L_gt_450cm_list[3]
cross_section_a10GeV = custom_cs_list[4]*pb_fb*f_custom_cs_list_1000TeV *P_ME_L_gt_450cm_list[4]
cross_section_aME = custom_cs_list[5]*pb_fb

L = 150_000 # fb^{-1} = 150 ab^{-1}

import math
import numpy as np

def poisson_logcdf(n_obs, mean, rtol=1e-15):
    """Return log P(N <= n_obs | mean) using stable Poisson recurrences."""
    mean = float(mean)
    if mean < 0:
        raise ValueError("Poisson mean must be non-negative")
    if not isinstance(n_obs, (int, np.integer)) or n_obs < 0:
        raise ValueError("n_obs must be a non-negative integer")
    if mean == 0:
        return 0.0

    if n_obs < mean:
        log_p = -mean + n_obs * math.log(mean) - math.lgamma(n_obs + 1)
        relative_sum = 1.0
        term = 1.0
        n = n_obs
        while n > 0:
            term *= n / mean
            relative_sum += term
            n -= 1
            if term <= rtol * relative_sum:
                break
        return min(0.0, log_p + math.log(relative_sum))

    n = n_obs + 1
    log_p = -mean + n * math.log(mean) - math.lgamma(n + 1)
    relative_sum = 1.0
    term = 1.0
    while True:
        n += 1
        term *= mean / n
        relative_sum += term
        if term <= rtol * relative_sum:
            break
    upper_tail = math.exp(log_p + math.log(relative_sum))
    upper_tail = min(max(upper_tail, 0.0), 1.0)
    return -math.inf if upper_tail == 1.0 else math.log1p(-upper_tail)

def poisson_cls(s, b, n_obs, log_cl_b=None):
    """Exact one-channel Poisson CL_s without nuisance parameters."""
    s = float(s)
    b = float(b)
    if s < 0 or b < 0:
        raise ValueError("s and b must be non-negative")
    if log_cl_b is None:
        log_cl_b = poisson_logcdf(n_obs, b)
    log_cl_s_plus_b = poisson_logcdf(n_obs, s + b)
    log_cls = min(0.0, log_cl_s_plus_b - log_cl_b)
    return 0.0 if log_cls < -745.0 else math.exp(log_cls)

In [ ]:
import numpy as np
# 定义阈值范围
thresholds = np.linspace(min(min(a1MeV_pre), min(a10MeV_pre), min(a100MeV_pre), min(a1GeV_pre), min(a10GeV_pre)),
                         max(max(a1MeV_pre), max(a10MeV_pre), max(a100MeV_pre), max(a1GeV_pre), max(a10GeV_pre)), 100)

# 初始化列表
cls_values_a1MeV = []
cls_values_a10MeV = []
cls_values_a100MeV = []
cls_values_a1GeV = []
cls_values_a10GeV = []
n_obs_values = []
eff_signal_a1MeV = []
eff_signal_a10MeV = []
eff_signal_a100MeV = []
eff_signal_a1GeV = []
eff_signal_a10GeV = []
eff_background = []

for th in thresholds:
    # 计算信号 S_1 (a10)
    N_signal1_total = len(a1MeV_pre)
    N_signal1_right = np.sum(a1MeV_pre > th)
    sigma_signal1 = cross_section_a1MeV  # 请根据具体的 cross section 填充

    S1 = sigma_signal1 * L * (N_signal1_right / N_signal1_total)
    eff_signal_a1MeV.append(N_signal1_right / N_signal1_total)

    # 计算信号 S_2 (a30)
    N_signal2_total = len(a10MeV_pre)
    N_signal2_right = np.sum(a10MeV_pre > th)
    sigma_signal2 = cross_section_a10MeV  # 请根据具体的 cross section 填充

    S2 = sigma_signal2 * L * (N_signal2_right / N_signal2_total)
    eff_signal_a10MeV.append(N_signal2_right / N_signal2_total)

    # 计算信号 S_3 (a50)
    N_signal3_total = len(a100MeV_pre)
    N_signal3_right = np.sum(a100MeV_pre > th)
    sigma_signal3 = cross_section_a100MeV  # 请根据具体的 cross section 填充

    S3 = sigma_signal3 * L * (N_signal3_right / N_signal3_total)
    eff_signal_a100MeV.append(N_signal3_right / N_signal3_total)

    # 计算信号 S_4 (a70)
    N_signal4_total = len(a1GeV_pre)
    N_signal4_right = np.sum(a1GeV_pre > th)
    sigma_signal4 = cross_section_a1GeV  # 请根据具体的 cross section 填充

    S4 = sigma_signal4 * L * (N_signal4_right / N_signal4_total)
    eff_signal_a1GeV.append(N_signal4_right / N_signal4_total)

    # 计算信号 S_5 (a85)
    N_signal5_total = len(a10GeV_pre)
    N_signal5_right = np.sum(a10GeV_pre > th)
    sigma_signal5 = cross_section_a10GeV  # 请根据具体的 cross section 填充

    S5 = sigma_signal5 * L * (N_signal5_right / N_signal5_total)
    eff_signal_a10GeV.append(N_signal5_right / N_signal5_total)

    # 计算背景 B (aME)
    N_background_total = len(aME_pre)
    N_background_right = np.sum(aME_pre > th)
    sigma_background = cross_section_aME  # 请根据具体的 cross section 填充

    B = sigma_background * L * (N_background_right / N_background_total)

    eff_background_right = N_background_right / N_background_total
    eff_background.append(eff_background_right)

    # 单一 counting channel、无 nuisance parameter 的精确 Poisson CL_s
    # notebook 中没有真实观测数据，因此采用最接近 B 的整数背景伪观测。
    n_obs = int(np.floor(B + 0.5))
    n_obs_values.append(n_obs)
    log_cl_b = poisson_logcdf(n_obs, B)

    cls_values_a1MeV.append(poisson_cls(S1, B, n_obs, log_cl_b))
    cls_values_a10MeV.append(poisson_cls(S2, B, n_obs, log_cl_b))
    cls_values_a100MeV.append(poisson_cls(S3, B, n_obs, log_cl_b))
    cls_values_a1GeV.append(poisson_cls(S4, B, n_obs, log_cl_b))
    cls_values_a10GeV.append(poisson_cls(S5, B, n_obs, log_cl_b))

# CL_s 越小，排除能力越强；为每个信号寻找最小 CL_s。
min_CLs_a1MeV = np.nanmin(cls_values_a1MeV)
optimal_threshold_a1MeV_idx = np.nanargmin(cls_values_a1MeV)
optimal_threshold_a1MeV = thresholds[optimal_threshold_a1MeV_idx].item()

min_CLs_a10MeV = np.nanmin(cls_values_a10MeV)
optimal_threshold_a10MeV_idx = np.nanargmin(cls_values_a10MeV)
optimal_threshold_a10MeV = thresholds[optimal_threshold_a10MeV_idx].item()

min_CLs_a100MeV = np.nanmin(cls_values_a100MeV)
optimal_threshold_a100MeV_idx = np.nanargmin(cls_values_a100MeV)
optimal_threshold_a100MeV = thresholds[optimal_threshold_a100MeV_idx].item()

min_CLs_a1GeV = np.nanmin(cls_values_a1GeV)
optimal_threshold_a1GeV_idx = np.nanargmin(cls_values_a1GeV)
optimal_threshold_a1GeV = thresholds[optimal_threshold_a1GeV_idx].item()

min_CLs_a10GeV = np.nanmin(cls_values_a10GeV)
optimal_threshold_a10GeV_idx = np.nanargmin(cls_values_a10GeV)
optimal_threshold_a10GeV = thresholds[optimal_threshold_a10GeV_idx].item()
# 计算每个数据集的权重，用于归一化
weights_a10 = np.ones_like(a1MeV_pre) / len(a1MeV)
weights_a30 = np.ones_like(a10MeV_pre) / len(a10MeV)
weights_a50 = np.ones_like(a100MeV_pre) / len(a100MeV)
weights_a70 = np.ones_like(a1GeV_pre) / len(a1GeV)
weights_a85 = np.ones_like(a10GeV_pre) / len(a10GeV)
weights_aME = np.ones_like(aME_pre) / len(aME)

In [ ]:
# 画图
fig, (ax_main, ax_sub, ax_eff) = plt.subplots(3, 1, figsize=(5, 8), gridspec_kw={'height_ratios': [1, 1, 1]}, sharex=True)

# 原图 (主图)
ax_main.hist(a1MeV_pre, bins=50, color='black', histtype='step', fill=False, label=r'$m_{a} = 1$ MeV', weights=weights_a10)
ax_main.hist(a10MeV_pre, bins=50, color='red', histtype='step', fill=False, label=r'$m_{a} = 10$ MeV', weights=weights_a30)
ax_main.hist(a100MeV_pre, bins=50, color='blue', histtype='step', fill=False, label=r'$m_{a} = 100$ MeV', weights=weights_a50)
ax_main.hist(a1GeV_pre, bins=50, color='green', histtype='step', fill=False, label=r'$m_{a} = 0.8$ GeV', weights=weights_a70)
ax_main.hist(a10GeV_pre, bins=50, color='magenta', histtype='step', fill=False, label=r'$m_{a} = 10$ GeV', weights=weights_a85)
ax_main.hist(aME_pre, bins=50, histtype='step', color='darkorange', fill=False, label=r'$\text{mono}-\gamma$', linestyle='--', linewidth=1, weights=weights_aME)

ax_main.set_yscale('linear')
ax_main.set_ylabel('Normalized Events', fontsize=15)
ax_main.legend(ncol=2)

# 使用 -log10(CL_s) 将最优排除点显示为向上的峰。
strength_floor = 1e-12

def exclusion_strength(cls_values):
    cls_array = np.clip(np.asarray(cls_values, dtype=float), 1e-300, 1.0)
    return np.maximum(-np.log10(cls_array), strength_floor)

ax_sub.plot(thresholds, exclusion_strength(cls_values_a1MeV), color='black', label=r'$m_{a} = 1$ MeV')
ax_sub.plot(thresholds, exclusion_strength(cls_values_a10MeV), color='red', label=r'$m_{a} = 10$ MeV')
ax_sub.plot(thresholds, exclusion_strength(cls_values_a100MeV), color='blue', label=r'$m_{a} = 100$ MeV')
ax_sub.plot(thresholds, exclusion_strength(cls_values_a1GeV), color='green', label=r'$m_{a} = 0.8$ GeV')
ax_sub.plot(thresholds, exclusion_strength(cls_values_a10GeV), color='magenta', label=r'$m_{a} = 10$ GeV')

ax_sub.set_ylabel(r'$-\log_{10}(CL_s)$', fontsize=15)
ax_sub.grid(False)
ax_sub.set_yscale('log')
#ax_sub.set_ylim(100,400)
ax_sub.tick_params(axis='x', labelsize=12)
ax_sub.tick_params(axis='y', labelsize=12)
ax_sub.legend(ncol=2)
ax_sub.set_ylabel(r'$-\log_{10}(CL_s)$', fontsize=15)
ax_sub.grid(False)
ax_sub.legend(ncol=2)
ax_sub.text(
    0.2, 0.55,
    r'$\Lambda=1000~\mathrm{TeV}$' + '\n'
    r'$\mathcal{L}=150~\mathrm{ab}^{-1}$',
    transform=ax_sub.transAxes,
    ha='left',
    va='top',
    fontsize=12
)

# 鉴别效率图 (子图)
ax_eff.plot(thresholds, eff_signal_a1MeV, color='black', linestyle='-', label=r'$m_{a} = 1$ MeV')
ax_eff.plot(thresholds, eff_signal_a10MeV, color='red', linestyle='-', label=r'$m_{a} = 10$ MeV')
ax_eff.plot(thresholds, eff_signal_a100MeV, color='blue', linestyle='-', label=r'$m_{a} = 100$ MeV')
ax_eff.plot(thresholds, eff_signal_a1GeV, color='green', linestyle='-', label=r'$m_{a} = 0.8$ GeV')
ax_eff.plot(thresholds, eff_signal_a10GeV, color='magenta', linestyle='-', label=r'$m_{a} = 10$ GeV')
ax_eff.plot(thresholds, eff_background, color='darkorange', linestyle='--', label=r'$\text{mono}-\gamma$')

ax_eff.set_xlabel('Threshold', fontsize=15)
ax_eff.set_ylabel('Efficiency', fontsize=15)
ax_eff.grid(False)
ax_eff.legend(ncol=2)

# 调整每个子图的 X 和 Y 轴刻度字体大小
ax_main.tick_params(axis='x', labelsize=12)
ax_main.tick_params(axis='y', labelsize=12)
ax_eff.tick_params(axis='x', labelsize=12)
ax_eff.tick_params(axis='y', labelsize=12)

# 调整布局并保存
plt.tight_layout()
plt.savefig("Normalized_Events_with_CLs_and_Efficiency_ML_2.png")
plt.show()


In [ ]:
import csv
import os
import numpy as np
# 创建保存目录
output_dir = "ML_output"
os.makedirs(output_dir, exist_ok=True)

# 使用与显著度单元相同的一维阈值范围
all_signal_scores = np.concatenate([
    np.ravel(a1MeV_pre),
    np.ravel(a10MeV_pre),
    np.ravel(a100MeV_pre),
    np.ravel(a1GeV_pre),
    np.ravel(a10GeV_pre)
])
thresholds = np.linspace(all_signal_scores.min(), all_signal_scores.max(), 100)

# 初始化效率与逐阈值 CL_s 列表
eff_signal_a1MeV = []
eff_signal_a10MeV = []
eff_signal_a100MeV = []
eff_signal_a1GeV = []
eff_signal_a10GeV = []
eff_background = []
cls_values_a1MeV = []
cls_values_a10MeV = []
cls_values_a100MeV = []
cls_values_a1GeV = []
cls_values_a10GeV = []
n_obs_values = []

for th in thresholds:
    signal_eff_1 = np.sum(a1MeV_pre > th) / len(a1MeV_pre)
    S1 = cross_section_a1MeV * L * signal_eff_1
    eff_signal_a1MeV.append(signal_eff_1)

    signal_eff_2 = np.sum(a10MeV_pre > th) / len(a10MeV_pre)
    S2 = cross_section_a10MeV * L * signal_eff_2
    eff_signal_a10MeV.append(signal_eff_2)

    signal_eff_3 = np.sum(a100MeV_pre > th) / len(a100MeV_pre)
    S3 = cross_section_a100MeV * L * signal_eff_3
    eff_signal_a100MeV.append(signal_eff_3)

    signal_eff_4 = np.sum(a1GeV_pre > th) / len(a1GeV_pre)
    S4 = cross_section_a1GeV * L * signal_eff_4
    eff_signal_a1GeV.append(signal_eff_4)

    signal_eff_5 = np.sum(a10GeV_pre > th) / len(a10GeV_pre)
    S5 = cross_section_a10GeV * L * signal_eff_5
    eff_signal_a10GeV.append(signal_eff_5)

    background_eff = np.sum(aME_pre > th) / len(aME_pre)
    B = cross_section_aME * L * background_eff
    eff_background.append(background_eff)

    n_obs = int(np.floor(B + 0.5))
    n_obs_values.append(n_obs)
    log_cl_b = poisson_logcdf(n_obs, B)
    cls_values_a1MeV.append(poisson_cls(S1, B, n_obs, log_cl_b))
    cls_values_a10MeV.append(poisson_cls(S2, B, n_obs, log_cl_b))
    cls_values_a100MeV.append(poisson_cls(S3, B, n_obs, log_cl_b))
    cls_values_a1GeV.append(poisson_cls(S4, B, n_obs, log_cl_b))
    cls_values_a10GeV.append(poisson_cls(S5, B, n_obs, log_cl_b))

# 将效率和每个信号对应的 CL_s 写入 CSV
with open("results_2.csv", "w", newline="") as csvfile:
    csv_writer = csv.writer(csvfile)
    csv_writer.writerow([
        "Threshold",
        "Signal_Efficiency_1MeV",
        "Signal_Efficiency_10MeV",
        "Signal_Efficiency_100MeV",
        "Signal_Efficiency_1GeV",
        "Signal_Efficiency_10GeV",
        "Background_Efficiency",
        "CLs_1MeV",
        "CLs_10MeV",
        "CLs_100MeV",
        "CLs_1GeV",
        "CLs_10GeV"
    ])

    for i in range(len(thresholds)):
        csv_writer.writerow([
            float(thresholds[i]),
            float(eff_signal_a1MeV[i]),
            float(eff_signal_a10MeV[i]),
            float(eff_signal_a100MeV[i]),
            float(eff_signal_a1GeV[i]),
            float(eff_signal_a10GeV[i]),
            float(eff_background[i]),
            float(cls_values_a1MeV[i]),
            float(cls_values_a10MeV[i]),
            float(cls_values_a100MeV[i]),
            float(cls_values_a1GeV[i]),
            float(cls_values_a10GeV[i])
        ])

print("所有数据已写入 results_2.csv 文件。")
